In [1]:
function countHeads(n::Int)
  c::Int = 0
  for i=1:n
    c += rand(Bool)
  end
  c
end

countHeads (generic function with 1 method)

In [2]:
@time countHeads(2*10^9)/(2*10^9)

  2.766904 seconds


0.4999841475

In [ ]:
using Distributed

In [4]:
nprocs()

1

In [5]:
addprocs(1)

1-element Vector{Int64}:
 2

In [6]:
nprocs()

2

In [14]:
@everywhere function countHeads(n::Int)
  c::Int = 0
  for i=1:n
    c += rand(Bool)
  end
  c
end

In [8]:
a= @spawn countHeads(10^9)
b= @spawn countHeads(10^9)

Future(2, 1, 6, ReentrantLock(), nothing)

In [9]:
fetch(a)+fetch(b)

1000012850

In [10]:
@time let 
  a= @spawn countHeads(10^9)
  b= @spawn countHeads(10^9)
  fetch(a)+fetch(b)
end

  2.780347 seconds (723 allocations: 34.172 KiB)


1000003369

In [11]:
addprocs(2)

2-element Vector{Int64}:
 3
 4

In [12]:
nprocs()

4

In [15]:
@time let
    a = @spawn countHeads(5*10^8)
    b = @spawn countHeads(5*10^8)
    c = @spawn countHeads(5*10^8)
    d = @spawn countHeads(5*10^8)
    fetch(a)+fetch(b)+fetch(c)+fetch(d)
end

  1.649405 seconds (1.23 k allocations: 59.844 KiB, 0.18% compilation time)


999963579

In [25]:
@time let
  nheads = @distributed (+) for i = 1:2*10^10
    Int(rand(Bool))
  end
end

 12.788984 seconds (15.46 k allocations: 786.367 KiB, 0.10% compilation time)


9999929928

In [24]:
nprocs()

3

In [23]:
rmprocs(1)

┌ Warning: rmprocs: process 1 not removed
└ @ Distributed /Users/pstaab/.julia/juliaup/julia-1.11.0-rc1+0.aarch64.apple.darwin14/share/julia/stdlib/v1.11/Distributed/src/cluster.jl:1049


Task (done) @0x000000010c6595f0

In [6]:
procs()

4-element Vector{Int64}:
 1
 2
 3
 4

In [17]:
rmprocs(2,3,4)

Task (done) @0x00000001107dc790

In [22]:
addprocs()

10-element Vector{Int64}:
  8
  9
 10
 11
 12
 13
 14
 15
 16
 17

In [23]:
procs()

14-element Vector{Int64}:
  1
  5
  6
  7
  8
  9
 10
 11
 12
 13
 14
 15
 16
 17

In [25]:
@time let
  nheads = @distributed (+) for i = 1:2*10^10
    rand(Bool)
  end
end

  5.133267 seconds (15.81 k allocations: 839.656 KiB, 0.45% compilation time)


9999908250

In [20]:
addprocs(3)

3-element Vector{Int64}:
 5
 6
 7

In [26]:
include("PlayingCards.jl")
using .PlayingCards, Random, Distributed

In [ ]:
function countHands(trials::Int,f::Function)
  local deck=map(Card,1:52)
  local num_hands=0
  for i=1:trials
      shuffle!(deck)
      h = Hand(deck[1:5])
      if f(h)
          num_hands+=1
      end
  end
  num_hands
end

countHands (generic function with 1 method)

In [28]:
@time countHands(10_000_000,isFullHouse)

  3.735101 seconds (60.00 M allocations: 3.129 GiB, 9.94% gc time)


14538

In [1]:
using Distributed

In [2]:
addprocs(3)

3-element Vector{Int64}:
 2
 3
 4

In [ ]:
@everywhere include("PlayingCards.jl")
@everywhere using .PlayingCards, Random

@everywhere function paraCountHands(trials::Integer,f::Function)
  local deck=map(Card,1:52)
  function checkHand(f::Function) ## shuffle the deck then check the hand.
    shuffle!(deck)
    f(Hand(deck[1:5]))
  end
  @distributed (+) for i = 1:trials
    Int(checkHand(f))
  end
end

In [6]:
@time fh = paraCountHands(10_000_000,isFullHouse)

  1.301482 seconds (409 allocations: 24.375 KiB)


14492

In [1]:
using Distributed

# 1. Add worker processes FIRST
if nprocs() == 1
    addprocs(3)
end

# 2. NOW distribute the file and module loading across all workers
@everywhere include("PlayingCards.jl")
@everywhere using .PlayingCards, Random

# 3. Define the parallel function on all workers
@everywhere function paraCountHands(trials::Integer, f::Function)
    # Thread-safe deck initialization inside the function
    deck = map(Card, 1:52)
    
    function checkHand(f::Function)
        # Note: shuffle! mutates local deck state
        # For full thread/task safety across parallel workers, 
        # ensure each worker/task shuffles its own local array instance.
        local_deck = copy(deck)
        shuffle!(local_deck)
        return f(Hand(local_deck[1:5]))
    end

    @distributed (+) for i = 1:trials
        Int(checkHand(f))
    end
end

# 4. Execute timing test
@time fh = paraCountHands(10_000_000, isFullHouse)

  4.694770 seconds (878.02 k allocations: 44.171 MiB, 0.82% gc time, 15.54% compilation time)


14175